In [ ]:
!pip install -q pyvinecopulib==1.0.0 numpy==2.1.3 scipy==1.16.3 \
              scikit-learn==1.6.1 pandas==2.2.3 matplotlib==3.10.0 \
              seaborn==0.13.2 openpyxl==3.1.5
!git clone https://github.com/mohsenbenhassine/ls-vine.git
%cd ls-vine
import sys
sys.path.insert(0, ".")

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

from src.config import CFG
from src.models import LSVineNet
from src.losses import (soft_tau_loss, rank_dependence_loss, reg_loss,
                         lambda_warmup)
from src.vine_utils import empirical_pit, fit_vine, vine_metrics
from src.datasets import make_student_dvine, make_mixed_rvine, split, split_real
from src.benchmark import select_d_lat
from src.train import set_seed, DEVICE, USE_AMP

# Ablation configurations
ABLATIONS = {
    "LS-Vine-Full":  {"use_soft_tau": True,  "use_rank_dep": True,
                      "use_reg": True,  "use_warmup": True},
    "A1-No-RankZ":   {"use_soft_tau": True,  "use_rank_dep": False,
                      "use_reg": True,  "use_warmup": False},
    "A2-No-SoftTau": {"use_soft_tau": False, "use_rank_dep": True,
                      "use_reg": True,  "use_warmup": True},
    "A3-No-Reg":     {"use_soft_tau": True,  "use_rank_dep": True,
                      "use_reg": False, "use_warmup": True},
    "A4-No-Warmup":  {"use_soft_tau": True,  "use_rank_dep": True,
                      "use_reg": True,  "use_warmup": False},
    "A5-MSE-Only":   {"use_soft_tau": False, "use_rank_dep": False,
                      "use_reg": False, "use_warmup": False},
}


def train_lsvine_ablation(X_tr, X_vl, d_lat, cfg, seed, ablation):
    """Train LS-Vine with a given ablation configuration."""
    ab = ABLATIONS[ablation]
    set_seed(seed)
    gen = torch.Generator(device="cpu"); gen.manual_seed(seed)

    d = X_tr.shape[1]
    model = LSVineNet(d, d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs,
                                                  eta_min=1e-5)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    curr_vine = best_vine = None
    best_aic = np.inf; best_state = None; patience = 0

    for ep in range(1, cfg.epochs + 1):
        in_pre = ep <= cfg.pretrain_epochs

        if not ab["use_rank_dep"]:
            lam = 0.0
        elif ab["use_warmup"]:
            lam = 0.0 if in_pre else lambda_warmup(
                ep - cfg.pretrain_epochs, cfg.lam_max, cfg.tau_w)
        else:
            lam = 0.0 if in_pre else cfg.lam_max

        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), cfg.batch_size):
            xb = Xtr[perm[i:i + cfg.batch_size]]
            z, xhat = model(xb)

            Lr = F.mse_loss(xhat, xb)
            if ab["use_soft_tau"]:
                Lr = Lr + cfg.alpha * soft_tau_loss(
                    xb, xhat, cfg.beta_tau, cfg.frac_pairs, gen)

            Lg = (reg_loss(z, cfg.lam_var) if ab["use_reg"]
                  else torch.zeros((), device=DEVICE))

            Lv = torch.zeros((), device=DEVICE)
            if ab["use_rank_dep"] and lam > 0 and not in_pre:
                Lv = rank_dependence_loss(
                    xb.float(), z.float(), beta=cfg.beta_tau,
                    frac_pairs=cfg.frac_pairs, gen=gen,
                    n_quantiles=cfg.n_quantiles, tail_weight=cfg.tail_weight)
                if not torch.isfinite(Lv):
                    Lv = torch.zeros((), device=DEVICE)

            loss = Lr + lam * Lv + cfg.gamma * Lg
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        if ep % cfg.K == 0 and not in_pre:
            model.eval()
            with torch.no_grad():
                Zf, _ = model(Xtr); Zf = Zf.float()
                Uf = empirical_pit(Zf.cpu().numpy()).astype(np.float64)
            try:
                curr_vine = fit_vine(Uf, cfg.trunc)
            except Exception:
                pass

        model.eval(); val_aic = np.nan
        if curr_vine is not None:
            with torch.no_grad():
                Zv, _ = model(Xvl)
                Uv = empirical_pit(Zv.float().cpu().numpy()).astype(np.float64)
            try:
                _, val_aic = vine_metrics(curr_vine, Uv)
            except Exception:
                pass

        if (ep > cfg.pretrain_epochs and curr_vine is not None
                and np.isfinite(val_aic)):
            if val_aic < best_aic:
                best_aic = val_aic
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
                best_vine = curr_vine
                patience = 0
            else:
                patience += 1
            if patience >= cfg.patience:
                break
        else:
            patience = 0

    if best_state:
        model.load_state_dict({k: v.to(DEVICE)
                               for k, v in best_state.items()})
    return model, best_vine


def run_ablation(seeds=None, scenarios=("S1", "S3"), cfg=CFG):
    """Run ablation on S1 and S3."""
    seeds = seeds or cfg.seeds
    SCENARIOS_ABL = {
        "S1": dict(builder=lambda s: make_student_dvine(10, 0.4, 4, 3500, s),
                   split_fn=split),
        "S3": dict(builder=lambda s: make_mixed_rvine(12, 3500, s),
                   split_fn=split),
    }

    rows = []
    for sc_key in scenarios:
        sc = SCENARIOS_ABL[sc_key]
        for seed in seeds:
            print(f"\n[{sc_key}] seed={seed}")
            ds = sc["split_fn"](sc["builder"](seed))
            d_lat = select_d_lat(ds["X_train"], cfg.d_lat_var_target)

            for ab_name in ABLATIONS:
                print(f"  -> {ab_name}", end=" ", flush=True)
                model, vine = train_lsvine_ablation(
                    ds["X_train"], ds["X_val"], d_lat, cfg, seed, ab_name)
                if vine is None:
                    print("FAIL")
                    continue

                Xts_t = torch.tensor(ds["X_test"], dtype=torch.float32).to(DEVICE)
                model.eval()
                with torch.no_grad():
                    Zts, Xhat = model(Xts_t)
                    Uts = empirical_pit(
                        Zts.float().cpu().numpy()).astype(np.float64)
                    Xhat_np = Xhat.float().cpu().numpy()

                ll, aic = vine_metrics(vine, Uts)
                from src.vine_utils import kendall_matrix
                tau_orig = kendall_matrix(ds["X_test"])
                tau_rec = kendall_matrix(Xhat_np)
                dep_rec = np.linalg.norm(tau_orig - tau_rec, "fro")

                rows.append({"scenario": sc_key, "seed": seed,
                             "ablation": ab_name, "LL": ll, "AIC": aic,
                             "DepRec": dep_rec})
                print(f"OK (AIC={aic:.2f})")

    return pd.DataFrame(rows)


df_ablation = run_ablation(seeds=CFG.seeds, scenarios=("S1", "S3"), cfg=CFG)
df_ablation.to_csv("results/ablation.csv", index=False)
print("\nSaved: results/ablation.csv")
print(df_ablation.groupby(["scenario", "ablation"])[["AIC", "DepRec"]]
      .agg(["mean", "std"]).round(3))